In [1]:
import re
import numpy as np
import random


In [2]:
data_path = "human_text.txt"
data_path2 = "robot_text.txt"

with open(data_path, 'r', encoding='utf-8') as f:
    lines = f.read().split('\n')

with open(data_path2, 'r', encoding='utf-8') as f:
    lines2 = f.read().split('\n')

lines = [re.sub(r"\[\w+\]", '', line) for line in lines]
lines = [" ".join(re.findall(r"\w+", line)) for line in lines]

lines2 = [re.sub(r"\[\w+\]", '', line) for line in lines2]
lines2 = [" ".join(re.findall(r"\w+", line)) for line in lines2]

pairs = list(zip(lines, lines2))


In [3]:
input_docs = []
target_docs = []
input_tokens = set()
target_tokens = set()

for line in pairs[:400]:
    input_doc, target_doc = line
    input_docs.append(input_doc)

    target_doc = "<START> " + target_doc + " <END>"
    target_docs.append(target_doc)

    for token in re.findall(r"[\w']+|[^\s\w]", input_doc):
        input_tokens.add(token)

    for token in target_doc.split():
        target_tokens.add(token)

input_tokens = sorted(list(input_tokens))
target_tokens = sorted(list(target_tokens))

num_encoder_tokens = len(input_tokens)
num_decoder_tokens = len(target_tokens)

input_features_dict = {token: i for i, token in enumerate(input_tokens)}
target_features_dict = {token: i for i, token in enumerate(target_tokens)}

reverse_input_features_dict = {i: token for token, i in input_features_dict.items()}
reverse_target_features_dict = {i: token for token, i in target_features_dict.items()}

max_encoder_seq_length = max([len(re.findall(r"[\w']+|[^\s\w]", doc)) for doc in input_docs])
max_decoder_seq_length = max([len(doc.split()) for doc in target_docs])

encoder_input_data = np.zeros((len(input_docs), max_encoder_seq_length, num_encoder_tokens))
decoder_input_data = np.zeros((len(input_docs), max_decoder_seq_length, num_decoder_tokens))
decoder_target_data = np.zeros((len(input_docs), max_decoder_seq_length, num_decoder_tokens))

for i, (input_doc, target_doc) in enumerate(zip(input_docs, target_docs)):
    for t, token in enumerate(re.findall(r"[\w']+|[^\s\w]", input_doc)):
        encoder_input_data[i, t, input_features_dict[token]] = 1.

    for t, token in enumerate(target_doc.split()):
        decoder_input_data[i, t, target_features_dict[token]] = 1.
        if t > 0:
            decoder_target_data[i, t-1, target_features_dict[token]] = 1.


In [4]:

from tensorflow import keras
from keras.layers import Input, LSTM, Dense
from keras.models import Model

dim = 256
batch_size = 10
epochs = 10   # reduce for testing

encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder_lstm = LSTM(dim, return_state=True)
_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)

decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

training_model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
training_model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])

training_model.fit([encoder_input_data, decoder_input_data], decoder_target_data,
                   batch_size=batch_size, epochs=epochs, validation_split=0.2)

training_model.save("training_model.h5")


Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 18s 415ms/step - accuracy: 0.0203 - loss: 1.5120 - val_accuracy: 0.0228 - val_loss: 1.4787
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 387ms/step - accuracy: 0.0257 - loss: 1.3192 - val_accuracy: 0.0215 - val_loss: 1.4609
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 389ms/step - accuracy: 0.0261 - loss: 1.2979 - val_accuracy: 0.0217 - val_loss: 1.4630
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 386ms/step - accuracy: 0.0259 - loss: 1.2929 - val_accuracy: 0.0215 - val_loss: 1.4671
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 366ms/step - accuracy: 0.0261 - loss: 1.2897 - val_accuracy: 0.0217 - val_loss: 1.4668
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 14s 438ms/step - accuracy: 0.0261 - loss: 1.2871 - val_accuracy: 0.0217 - val_loss: 1.4648
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 417ms/step - accuracy: 0.0259 - loss: 1.2844 - val_accuracy: 0.0217 - val_loss: 1.4701
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 379ms/step - accuracy: 0.0260 - loss: 1.2840 - val_accu

In [5]:
from keras.models import load_model

training_model = load_model("training_model.h5")

encoder_inputs = training_model.input[0]
encoder_outputs, state_h_enc, state_c_enc = training_model.layers[2].output
encoder_model = Model(encoder_inputs, [state_h_enc, state_c_enc])

latent_dim = 256
decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

decoder_outputs_inf, state_h_inf, state_c_inf = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs)

decoder_outputs_inf = decoder_dense(decoder_outputs_inf)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf, state_h_inf, state_c_inf]
)


In [6]:
def decode_response(input_matrix):
    states_value = encoder_model.predict(input_matrix)

    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, target_features_dict["<START>"]] = 1.

    decoded_sentence = ""

    for _ in range(20):
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)

        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_token = reverse_target_features_dict[sampled_token_index]

        if sampled_token == "<END>":
            break

        decoded_sentence += " " + sampled_token

        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.

        states_value = [h, c]

    return decoded_sentence.strip()


In [7]:
class ChatBot:
    def string_to_matrix(self, user_input):
        tokens = re.findall(r"[\w']+|[^\s\w]", user_input)
        mat = np.zeros((1, max_encoder_seq_length, num_encoder_tokens))
        for t, token in enumerate(tokens):
            if token in input_features_dict:
                mat[0, t, input_features_dict[token]] = 1.
        return mat

    def generate_response(self, text):
        mat = self.string_to_matrix(text)
        return decode_response(mat)

chatbot = ChatBot()


In [11]:
def ask_bot(text):
    return chatbot.generate_response(text)

ask_bot("hi")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step


'i i'

In [13]:
hi


NameError: name 'hi' is not defined

In [14]:
ask_bot("hi")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step


'i i'

In [15]:
ask_bot("hello")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step


'i i'

In [16]:
ask_bot("how are you")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step


'i i'

In [17]:
ask_bot("what is your name")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step


'i i'